# Iris Classifier using Vertex AI


## Overview

In this tutorial, you build a scikit-learn model and deploy it on infer in local environment using Google Cloud Storage for logging and tracking model and data


### Dataset

This tutorial uses R.A. Fisher's Iris dataset, a small and popular dataset for machine learning experiments. Each instance has four numerical features, which are different measurements of a flower, and a target label that
categorizes the flower into: **Iris setosa**, **Iris versicolour** and **Iris virginica**.

This tutorial uses [a version of the Iris dataset available in the
scikit-learn library](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html#sklearn.datasets.load_iris).

### Costs

This tutorial uses billable components of Google Cloud:

* Vertex AI
* Cloud Storage

Learn about [Vertex AI
pricing](https://cloud.google.com/vertex-ai/pricing), [Cloud Storage
pricing](https://cloud.google.com/storage/pricing), 

## Get started

### Install Vertex AI SDK for Python and other required packages



In [1]:

# Vertex SDK for Python
! pip3 install --upgrade --quiet  google-cloud-aiplatform


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


### Set Google Cloud project information
Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
PROJECT_ID = "bamboo-bulwark-473016-b4"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

In [3]:
BUCKET_URI = f"gs://mlops-course-bamboo-bulwark-473016-b4-week1"  # @param {type:"string"}
MODEL_ARTIFACT_DIR="iris_classifier/model"

**If your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [4]:
! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

Creating gs://mlops-course-bamboo-bulwark-473016-b4-week1/...
ServiceException: 409 A Cloud Storage bucket named 'mlops-course-bamboo-bulwark-473016-b4-week1' already exists. Try another name. Bucket names must be globally unique across all Google Cloud projects, including those outside of your organization.


### Initialize Vertex AI SDK for Python

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

In [5]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

/opt/conda/lib/python3.10/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


### Import the required libraries

In [6]:
import os
import sys

# Configure resource names

MODEL_ARTIFACT_DIR - Folder directory path to your model artificates within a Cloud Storage bucket.

REPOSITORY - Name of the Artificat Repository to create or use.

IMAGE - Name of the container image that is pushed to the repository.

MODEL_DISPLAY_NAME - Display name of the Vertex AI model resource.

In [7]:
MODEL_ARTIFACT_DIR = "my-models/iris-classifier-week-1"
REPOSITORY = "iris-classifier-repo"
IMAGE = "iris-classifier-img"
MODEL_DISPLAY_NAME = "iris-classifier"

# MLflow setup

In [8]:
import mlflow
from mlflow import MlflowClient
from mlflow.models import infer_signature
from pprint import pprint

mlflow.set_tracking_uri("http://127.0.0.1:8100")
client = MlflowClient(mlflow.get_tracking_uri())
all_experiments = client.search_experiments()
pprint(all_experiments)

[<Experiment: artifact_location='mlflow-artifacts:/773226422953504865', creation_time=1761943196119, experiment_id='773226422953504865', last_update_time=1761943196119, lifecycle_stage='active', name='IRIS_classifier_ga_week_5_test', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/709176548317874359', creation_time=1761940085374, experiment_id='709176548317874359', last_update_time=1761940085374, lifecycle_stage='active', name='Test', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/801345223767237230', creation_time=1761940058795, experiment_id='801345223767237230', last_update_time=1761940058795, lifecycle_stage='active', name='IRIS_classifier_ga_week_5', tags={'mlflow.experimentKind': 'custom_model_development'}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1761936430641, experiment_id='0', last_update_time=1761936430641, lifecycle_stage='active', name='Default', tags={}>]


In [9]:
mlflow.get_tracking_uri()

'http://127.0.0.1:8100'

In [42]:
mlflow.set_experiment("IRIS_classifier_ga_week_5_demo")

2025/10/31 21:23:07 INFO mlflow.tracking.fluent: Experiment with name 'IRIS_classifier_ga_week_5_demo' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/102262737435667917', creation_time=1761945787124, experiment_id='102262737435667917', last_update_time=1761945787124, lifecycle_stage='active', name='IRIS_classifier_ga_week_5_demo', tags={}>

In [43]:
all_experiments = client.search_experiments()
pprint(all_experiments)

[<Experiment: artifact_location='mlflow-artifacts:/102262737435667917', creation_time=1761945787124, experiment_id='102262737435667917', last_update_time=1761945787124, lifecycle_stage='active', name='IRIS_classifier_ga_week_5_demo', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/773226422953504865', creation_time=1761943196119, experiment_id='773226422953504865', last_update_time=1761943196119, lifecycle_stage='active', name='IRIS_classifier_ga_week_5_test', tags={'mlflow.experimentKind': 'custom_model_development'}>,
 <Experiment: artifact_location='mlflow-artifacts:/709176548317874359', creation_time=1761940085374, experiment_id='709176548317874359', last_update_time=1761940085374, lifecycle_stage='active', name='Test', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/801345223767237230', creation_time=1761940058795, experiment_id='801345223767237230', last_update_time=1761940058795, lifecycle_stage='active', name='IRIS_classifier_ga_week_5', tags={'mlflow.

## Simple Decision Tree model
Build a Decision Tree model on iris data

In [44]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pandas.plotting import parallel_coordinates
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import metrics
import gcsfs   

# --- 1️⃣ Define bucket and file path ---
file_path = "data/iris.csv"   # e.g. "datasets/iris.csv"
gcs_uri = f"gs://{BUCKET_URI}/{file_path}"

# --- 2️⃣ Read CSV directly from GCS ---
data = pd.read_csv(gcs_uri, storage_options={"token": "cloud"})  
# or storage_options={"token": None} if it's public

# --- 3️⃣ Quick check ---
print(data.head(5))


   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa


In [45]:
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

In [51]:
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics

# Define hyperparameters
params = {
    "max_depth": 3,
    "random_state": 1
}

# Train model
mod_dt = DecisionTreeClassifier(**params)
mod_dt.fit(X_train, y_train)

# Predict
prediction = mod_dt.predict(X_test)

# Evaluate accuracy
accuracy_score = metrics.accuracy_score(y_test, prediction)
print("The accuracy of the Decision Tree is: {:.3f}".format(accuracy_score))

The accuracy of the Decision Tree is: 0.983


In [52]:
import pickle
import joblib

joblib.dump(mod_dt, "artifacts/model.joblib")

['artifacts/model.joblib']

### Upload model artifacts and custom code to Cloud Storage

Before you can deploy your model for serving, Vertex AI needs access to the following files in Cloud Storage:

* `model.joblib` (model artifact)
* `preprocessor.pkl` (model artifact)

Run the following commands to upload your files:

In [53]:
import datetime

timestamp = datetime.datetime.utcnow().strftime("%Y%m%d-%H%M%S")

# Copy model into a timestamped subfolder
!gsutil cp artifacts/model.joblib {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/{timestamp}/model.joblib


Copying file://artifacts/model.joblib [Content-Type=application/octet-stream]...
/ [1 files][  2.5 KiB/  2.5 KiB]                                                
Operation completed over 1 objects/2.5 KiB.                                      


# Log the run to MLflow

In [54]:
from mlflow.models import infer_signature
import mlflow
import mlflow.sklearn

# Start MLflow run
with mlflow.start_run():
    # Log parameters and metrics
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", accuracy_score)

    # Add a tag (metadata)
    mlflow.set_tag("Training Info", "Decision tree model for IRIS data")

    # Infer model signature for schema tracking
    signature = infer_signature(X_train, mod_dt.predict(X_train))

    # Log the trained model to MLflow
    model_info = mlflow.sklearn.log_model(
        sk_model=mod_dt,
        artifact_path="iris_model",
        signature=signature,
        input_example=X_train,
        registered_model_name="IRIS-classifier-dt_demo",
    )


2025/10/31 21:27:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'IRIS-classifier-dt_demo' already exists. Creating a new version of this model...
2025/10/31 21:27:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: IRIS-classifier-dt_demo, version 2


🏃 View run painted-moth-752 at: http://127.0.0.1:8100/#/experiments/102262737435667917/runs/16023bb1f4fb47d3a2423107db378399
🧪 View experiment at: http://127.0.0.1:8100/#/experiments/102262737435667917


Created version '2' of model 'IRIS-classifier-dt_demo'.


# Inference Setup

In [57]:
import os, time, json, pandas as pd
import mlflow, mlflow.pyfunc
from mlflow.tracking import MlflowClient
from sklearn import metrics
import gcsfs

# ---- Tracking / Registry ----
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:8100")
EXPERIMENT_NAME     = "IRIS_classifier_ga_week_5_demo"
MODEL_NAME          = "IRIS-classifier-dt_demo"
ARTIFACT_SUBPATH    = "iris_model"        # artifact_path used when logging models
REGISTRY_STAGE      = "Production"

# ---- Data / Output ----
PROJECT_ID   = "bamboo-bulwark-473016"
BUCKET_ROOT  = "gs://mlops-course-bamboo-bulwark-473016-b4-week1"
EVAL_GCS_CSV = f"{BUCKET_ROOT}/data/iris_v1.csv"

ARTIFACTS_PREFIX = f"{BUCKET_ROOT}/my-models/iris-classifier-week-1"
OUTPUT_PREFIX    = "infer_eval"           # used for output filenames

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()


# Find best run (by accuracy) and REGISTER its model

In [58]:
exp = client.get_experiment_by_name(EXPERIMENT_NAME)
assert exp is not None, f"Experiment not found: {EXPERIMENT_NAME}"

runs = client.search_runs(
    [exp.experiment_id],
    order_by=["metrics.accuracy DESC"],
    max_results=1
)
assert runs, "No runs with metric 'accuracy' found"
best_run = runs[0]
best_run_id = best_run.info.run_id
best_acc = best_run.data.metrics.get("accuracy")

best_model_uri = f"runs:/{best_run_id}/{ARTIFACT_SUBPATH}"
print("🏆 Best run selected:")
print(f"• Run ID     : {best_run_id}")
print(f"• Accuracy   : {best_acc:.4f}")
print(f"• Model path : {best_model_uri}")
mv = mlflow.register_model(model_uri=best_model_uri, name=MODEL_NAME)
print("Registered version:", mv.version)


Registered model 'IRIS-classifier-dt_demo' already exists. Creating a new version of this model...
2025/10/31 21:29:19 WARNING mlflow.tracking._model_registry.fluent: Run with id 16023bb1f4fb47d3a2423107db378399 has no artifacts at artifact path 'iris_model', registering model based on models:/m-4cac5c2897814c1883904cb5a97f56a9 instead
2025/10/31 21:29:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: IRIS-classifier-dt_demo, version 3


🏆 Best run selected:
• Run ID     : 16023bb1f4fb47d3a2423107db378399
• Accuracy   : 0.9833
• Model path : runs:/16023bb1f4fb47d3a2423107db378399/iris_model
Registered version: 3


Created version '3' of model 'IRIS-classifier-dt_demo'.


# Promote that version to Production

In [59]:
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=mv.version,
    stage=REGISTRY_STAGE,
    archive_existing_versions=True
)
print(f"{MODEL_NAME} v{mv.version} -> {REGISTRY_STAGE}")


IRIS-classifier-dt_demo v3 -> Production


/var/tmp/ipykernel_35559/1431577125.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


# Load Production model from Registry + load dataset

In [60]:
model_uri = f"models:/{MODEL_NAME}/{REGISTRY_STAGE}"
model = mlflow.pyfunc.load_model(model_uri)
print("Loaded:", model_uri)

df = pd.read_csv(EVAL_GCS_CSV, storage_options={"token": "cloud"})
print("Data shape:", df.shape)

target_col = "species" if "species" in df.columns else ("target" if "target" in df.columns else None)
y_true = df[target_col] if target_col in df.columns else None


Loaded: models:/IRIS-classifier-dt_demo/Production
Data shape: (101, 5)


# Resolve input columns (signature → feature_names_in_ → fallback)

In [61]:
input_cols = None

# Try schema from model signature (recommended)
try:
    schema = model.metadata.get_input_schema()
    if schema and schema.inputs:
        input_cols = [c.name for c in schema.inputs]
except Exception:
    pass

# Legacy signature access
if input_cols is None:
    try:
        input_cols = list(model.metadata.signature.inputs.input_names())
    except Exception:
        pass

# sklearn fallback
if input_cols is None and hasattr(model, "feature_names_in_"):
    input_cols = list(model.feature_names_in_)

# Last resort: all columns except target
if input_cols is None:
    input_cols = [c for c in df.columns if c != target_col]

missing = [c for c in input_cols if c not in df.columns]
if missing:
    raise ValueError(f"Eval data missing required columns: {missing}")

print("Using features:", input_cols)


Using features: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']


# Predict and write outputs to GCS

In [62]:
import joblib

# ---------- Predict ----------
y_pred = model.predict(df[input_cols])

# ---------- Prepare output paths ----------
fs = gcsfs.GCSFileSystem(token="cloud")
ts = int(time.time())
base = f"{ARTIFACTS_PREFIX}/inference/{OUTPUT_PREFIX}_registry_{ts}"
preds_uri  = f"{base}_preds.csv"
report_uri = f"{base}_report.json"
model_uri_out = f"{base}_model.joblib"

# ---------- Save model ----------
# Save locally first
joblib.dump(model, "model.joblib")

# Upload model to GCS
with fs.open(model_uri_out, "wb") as f:
    f.write(open("model.joblib", "rb").read())

# ---------- Save predictions ----------
out = df.copy()
out["prediction"] = y_pred
with fs.open(preds_uri, "w") as f:
    out.to_csv(f, index=False)

# ---------- Create and save report ----------
report = {
    "model_uri": model_uri,
    "eval_data": EVAL_GCS_CSV,
    "n_rows": int(out.shape[0]),
    "has_ground_truth": bool(y_true is not None),
    "input_cols": input_cols,
    "model_saved_to": model_uri_out,
}

if y_true is not None:
    report["accuracy"] = float(metrics.accuracy_score(y_true, y_pred))
    report["macro_f1"]  = float(metrics.f1_score(y_true, y_pred, average="macro"))

with fs.open(report_uri, "w") as f:
    f.write(json.dumps(report, indent=2))

# ---------- Output summary ----------
print("✅ Model saved to:", model_uri_out)
print("✅ Predictions saved to:", preds_uri)
print("✅ Report saved to:", report_uri)
print("Model used:", model_uri)


✅ Model saved to: gs://mlops-course-bamboo-bulwark-473016-b4-week1/my-models/iris-classifier-week-1/inference/infer_eval_registry_1761946212_model.joblib
✅ Predictions saved to: gs://mlops-course-bamboo-bulwark-473016-b4-week1/my-models/iris-classifier-week-1/inference/infer_eval_registry_1761946212_preds.csv
✅ Report saved to: gs://mlops-course-bamboo-bulwark-473016-b4-week1/my-models/iris-classifier-week-1/inference/infer_eval_registry_1761946212_report.json
Model used: models:/IRIS-classifier-dt_demo/Production
